In [6]:
!pip install pandas scikit-learn nltk -q

In [1]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ================================
# 2. LOAD DATASET
# ================================
df = pd.read_csv('/kaggle/input/aitxtdetector/ai_human_detection_v1.csv')

TEXT_COL = "text"
LABEL_COL = "human_or_ai"

print("Dataset size:", len(df))
print(df[LABEL_COL].value_counts())


# ================================
# 3. CLEAN TEXT FUNCTION
# ================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df[TEXT_COL].apply(clean_text)


# ================================
# 4. TRAIN TEST SPLIT
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df[LABEL_COL],
    test_size=0.2,
    random_state=42,
    stratify=df[LABEL_COL]
)


# ================================
# 5. TF-IDF FEATURE EXTRACTION (TUNED)
# ================================
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,4),
    stop_words="english",
    min_df=2,
    sublinear_tf=True
)
def extra_features(text):
    words = text.split()
    return [
        len(words),
        np.mean([len(w) for w in words]) if words else 0,
        text.count("."),
        text.count(","),
        text.count("!")
    ]

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# ================================
# 6. MODEL TRAINING (BALANCED SVM)
# ================================
model = LinearSVC(
    class_weight={
        "ai": 1,
        "human": 3,
        "post_edited_ai": 2
    }
)

model.fit(X_train_vec, y_train)


# ================================
# 7. CROSS VALIDATION (ROBUSTNESS)
# ================================
scores = cross_val_score(model, X_train_vec, y_train, cv=5)
print("Cross-val accuracy:", scores.mean())


# ================================
# 8. MODEL EVALUATION
# ================================
preds = model.predict(X_test_vec)

print("\nAccuracy:", accuracy_score(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))


# ================================
# 9. CUSTOM TEXT PREDICTION
# ================================
def predict_text(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

print("\nSample Prediction:")
print(predict_text("AI is definitely changing industries by automating routine work and improving efficiency. While many companies are adopting these tools to stay competitive, I think we still need to be careful about ethical issues and data privacy."))


# ================================
# 10. SAVE MODEL
# ================================
pickle.dump(model, open("final_ai_text_model.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))

print("\nModel saved successfully!")

Dataset size: 686
human_or_ai
ai                335
human             181
post_edited_ai    170
Name: count, dtype: int64
Cross-val accuracy: 0.6641701417848207

Accuracy: 0.7318840579710145

Classification Report:
                 precision    recall  f1-score   support

            ai       0.69      0.88      0.77        67
         human       1.00      0.78      0.88        37
post_edited_ai       0.57      0.38      0.46        34

      accuracy                           0.73       138
     macro avg       0.75      0.68      0.70       138
  weighted avg       0.74      0.73      0.72       138


Confusion Matrix:
 [[59  0  8]
 [ 6 29  2]
 [21  0 13]]

Sample Prediction:
ai

Model saved successfully!


In [5]:
print(predict_text("ai is dead"))

ai
